In [1]:
%load_ext ElasticNotebook

Enabled rmm statistics


In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_19_try_0.pickle

In [3]:
%%RecordEvent
%%time
### cell 20 ###

def add_gap_rows_2(essay):
    # select only the relevant columns for this essay
    cols_to_keep = ['discourse_start', 'discourse_end', 'discourse_type', 'gap_length', 'gap_end_length']
    df = train.loc[train['id'] == essay, cols_to_keep].reset_index(drop=True)
    # preserve original print
    print(df)

    # vectorized computation of interior gaps
    starts = df['discourse_end'].shift().add(1)
    ends = df['discourse_start'].sub(1)
    mask = (df['gap_length'] > 0) & (df.index >= 1)
    if mask.any():
        new_gaps = pd.DataFrame({
            'discourse_start': starts[mask],
            'discourse_end': ends[mask],
            'discourse_type': 'Nothing',
            'gap_length': np.nan,
            'gap_end_length': np.nan
        })
        df = pd.concat([df, new_gaps], ignore_index=True)

    # sort after inserting interior gaps
    df = df.sort_values('discourse_start').reset_index(drop=True)

    # add end gap if present (appended at bottom to match original semantics)
    if df['gap_end_length'].iloc[-1] > 0:
        start = df['discourse_end'].iloc[-1] + 1
        end = start + df['gap_end_length'].iloc[-1]
        df.loc[len(df)] = [start, end, 'Nothing', np.nan, np.nan]

    return df


def print_colored_essay(essay):
    df_essay = add_gap_rows_2(essay)
    # build ents list via vectorized to_dict
    ents = (
        df_essay[['discourse_start', 'discourse_end', 'discourse_type']]
        .rename(columns={
            'discourse_start': 'start',
            'discourse_end': 'end',
            'discourse_type': 'label'
        })
        .astype({'start': 'int', 'end': 'int'})
        .to_dict('records')
    )
    # read the essay text
    essay_file = (
        f"/home/dias-benchmarks/notebooks/erikbruin/"
        f"nlp-on-student-writing-eda/input/feedback-prize-2021/train/{essay}.txt"
    )
    with open(essay_file, 'r') as file:
        data = file.read()

    doc2 = {"text": data, "ents": ents}
    colors = {
        'Lead': '#EE11D0', 'Position': '#AB4DE1', 'Claim': '#1EDE71',
        'Evidence': '#33FAFA', 'Counterclaim': '#4253C1',
        'Concluding Statement': 'yellow', 'Rebuttal': 'red'
    }
    options = {"ents": df_essay.discourse_type.unique().tolist(), "colors": colors}


CPU times: user 3 µs, sys: 1 µs, total: 4 µs
Wall time: 7.39 µs


In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_20_try_0.pickle